In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

SNOWFLAKE_OPTIONS = {
    "sfUrl": "RCVOYBU-ZSC95331.snowflakecomputing.com",
    "sfUser": "IMDB_USER",
    "sfPassword": "IMDB",
    "sfDatabase": "IMDB_DB",
    "sfSchema": "DW",
    "sfWarehouse": "IMDB_WH",
    "sfRole": "IMDB_ROLE"
}

# Configuration
VOLUME_BASE = "/Volumes/imdb_final_project/raw/raw_store"
ENABLE_SCHEMA_EVOLUTION = False

# Helper function to reduce code duplication
def create_bronze_table(subfolder, schema_hints):
    """Helper to create bronze ingestion logic"""
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{VOLUME_BASE}/{subfolder}/_schema_checkpoint")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaHints", schema_hints)
            .option("delimiter", "\t")
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", "\"")
            .option("nullValue", "\\N")
    )
    
    if ENABLE_SCHEMA_EVOLUTION:
        reader = reader.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    
    df = reader.load(f"{VOLUME_BASE}/{subfolder}")
    
    # # Rename columns
    # for col_name in df.columns:
    #     clean_name = col_name.replace(" ", "_").replace("#", "Number")
    #     df = df.withColumnRenamed(col_name, clean_name)
    

    import re
    for col_name in df.columns:
        # Replace invalid chars with underscore, then clean up multiple underscores
        clean_name = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
        clean_name = clean_name.replace("#", "Number")
        # Remove leading/trailing underscores
        clean_name = clean_name.strip('_')
        df = df.withColumnRenamed(col_name, clean_name)

    # Add audit columns
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_date", current_date())
    )

# ============================================
# BRONZE TABLES
# ============================================

@dlt.table(
    name="bronze_name_basics_raw",
    comment="Raw IMDB name.basics data"
)
def bronze_name_basics():
    return create_bronze_table("name_basics", "nconst STRING, birthYear STRING, deathYear STRING")

@dlt.table(
    name="bronze_title_akas_raw",
    comment="Raw IMDB title.akas data"
)
def bronze_title_akas():
    return create_bronze_table("title_akas", "titleId STRING, ordering STRING")

@dlt.table(
    name = "bronze_title_crew_raw",
    comment = "Raw IMDB title.crew data"
)
def bronze_title_crew():
    return create_bronze_table("title_crew", "tconst STRING, directors STRING, writers STRING")

@dlt.table(
    name="bronze_title_ratings_raw",
    comment="Raw IMDB title.ratings data"
)
def bronze_title_ratings():
    return create_bronze_table("title_ratings", "tconst STRING, averageRating STRING, numVotes STRING")

@dlt.table(
    name = "bronze_title_region_raw",
    comment = "Raw IMDB title.region data"
)
def bronze_title_region():
    return create_bronze_table("title_region", "tconst STRING, region STRING"
)
    
@dlt.table(
    name="bronze_title_episode_raw",
    comment="Raw IMDB title.episode data"
)
def bronze_title_episode():
    return create_bronze_table("title_episode", "tconst STRING, parentTconst STRING, seasonNumber STRING, episodeNumber STRING")

@dlt.table(
    name="bronze_title_principals_raw",
    comment="Raw IMDB title.principals data"
)
def bronze_title_principals():
    return create_bronze_table("title_principals", "tconst STRING, ordering STRING")

@dlt.table(
    name="bronze_title_basics_raw",
    comment="Raw IMDB title.basics data"
)
def bronze_title_basics():
    return create_bronze_table("title_basics", "tconst STRING, titleType STRING, primaryTitle STRING, originalTitle STRING")


@dlt.table(
    name = "bronze_title_language_codes_raw",
    comment = "Raw IMDB title.language_codes data"
)

def bronze_title_language_codes():
    return create_bronze_table("language_codes", "Language_name STRING, language_code STRING")


In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_name_basics",
    comment="Cleaned IMDB name basics data - preserves all records without explosion",
    table_properties={
        "quality": "silver",    
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_nconst": "NCONST IS NOT NULL AND NCONST RLIKE '^nm[0-9]{7,}$'",
    # "valid_primary_name": "PRIMARY_NAME IS NOT NULL AND LENGTH(TRIM(PRIMARY_NAME)) > 0"
})
def silver_name_basics():
    df = dlt.read_stream("bronze_name_basics_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("nconst", "NCONST")
        .withColumnRenamed("primaryName", "PRIMARY_NAME")
        .withColumnRenamed("birthYear", "BIRTH_YEAR")
        .withColumnRenamed("deathYear", "DEATH_YEAR")
        .withColumnRenamed("primaryProfession", "PRIMARY_PROFESSION")
        .withColumnRenamed("knownForTitles", "KNOWN_FOR_TITLES")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("BIRTH_YEAR", 
                   when(col("BIRTH_YEAR").isNull(), "0000")
                   .otherwise(col("BIRTH_YEAR")))
        .withColumn("DEATH_YEAR", 
                   when(col("DEATH_YEAR").isNull(), "9999")
                   .otherwise(col("DEATH_YEAR")))
        .withColumn("PRIMARY_PROFESSION", 
                   when(col("PRIMARY_PROFESSION").isNull(), "Unknown")
                   .otherwise(col("PRIMARY_PROFESSION")))
        .withColumn("KNOWN_FOR_TITLES", 
                   when(col("KNOWN_FOR_TITLES").isNull(), "Unknown")
                   .otherwise(col("KNOWN_FOR_TITLES")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("BIRTH_YEAR", col("BIRTH_YEAR").cast("int"))
        .withColumn("DEATH_YEAR", col("DEATH_YEAR").cast("int"))
    )
    
    # Add is_alive flag
    df = df.withColumn("IS_ALIVE", when(col("DEATH_YEAR") == 9999, True).otherwise(False))
    
    # Trim whitespace
    df = (
        df
        .withColumn("PRIMARY_PROFESSION", trim(col("PRIMARY_PROFESSION")))
        .withColumn("KNOWN_FOR_TITLES", trim(col("KNOWN_FOR_TITLES")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "NCONST",
        "PRIMARY_NAME",
        "BIRTH_YEAR",
        "DEATH_YEAR",
        "IS_ALIVE",
        "PRIMARY_PROFESSION",
        "KNOWN_FOR_TITLES",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_title_akas",
    comment="Cleaned IMDB title.akas data with NULL handling - no explosion needed",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_titleId": "TITLE_ID IS NOT NULL AND TITLE_ID RLIKE '^tt[0-9]{7,}$'",
    "valid_ordering": "ORDERING >= -1"
})
def silver_title_akas():
    """
    Cleans the bronze title_akas data.
    
    Transformations:
    - Replace \\N with NULLs
    - Replace NULL ordering with -1
    - Replace NULL text fields with 'Unknown'
    - Replace NULL isOriginalTitle with -1
    - Create boolean flag for isOriginalTitle
    - Trim whitespace
    """
    
    df = dlt.read_stream("bronze_title_akas_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("titleId", "TITLE_ID")
        .withColumnRenamed("ordering", "ORDERING")
        .withColumnRenamed("title", "TITLE")
        .withColumnRenamed("region", "REGION")
        .withColumnRenamed("language", "LANGUAGE")
        .withColumnRenamed("types", "TYPES")
        .withColumnRenamed("attributes", "ATTRIBUTES")
        .withColumnRenamed("isOriginalTitle", "IS_ORIGINAL_TITLE")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("ORDERING", 
                   when(col("ORDERING").isNull(), "-1")
                   .otherwise(col("ORDERING")))
        .withColumn("TITLE", 
                   when(col("TITLE").isNull(), "Unknown")
                   .otherwise(col("TITLE")))
        .withColumn("REGION", 
                   when(col("REGION").isNull(), "Unknown")
                   .otherwise(col("REGION")))
        .withColumn("LANGUAGE", 
                   when(col("LANGUAGE").isNull(), "Unknown")
                   .otherwise(col("LANGUAGE")))
        .withColumn("TYPES", 
                   when(col("TYPES").isNull(), "Unknown")
                   .otherwise(col("TYPES")))
        .withColumn("ATTRIBUTES", 
                   when(col("ATTRIBUTES").isNull(), "Unknown")
                   .otherwise(col("ATTRIBUTES")))
        .withColumn("IS_ORIGINAL_TITLE", 
                   when(col("IS_ORIGINAL_TITLE").isNull(), "-1")
                   .otherwise(col("IS_ORIGINAL_TITLE")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("ORDERING", col("ORDERING").cast("int"))
        .withColumn("IS_ORIGINAL_TITLE", col("IS_ORIGINAL_TITLE").cast("int"))
    )
    
    # Create boolean flag for original title
    df = df.withColumn("IS_ORIGINAL_TITLE_FLAG",
                      when(col("IS_ORIGINAL_TITLE") == 1, True)
                      .when(col("IS_ORIGINAL_TITLE") == 0, False)
                      .otherwise(None))
    
    # Trim whitespace
    df = (
        df
        .withColumn("TITLE", trim(col("TITLE")))
        .withColumn("REGION", trim(col("REGION")))
        .withColumn("LANGUAGE", trim(col("LANGUAGE")))
        .withColumn("TYPES", trim(col("TYPES")))
        .withColumn("ATTRIBUTES", trim(col("ATTRIBUTES")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "TITLE_ID",
        "ORDERING",
        "TITLE",
        "REGION",
        "LANGUAGE",
        "TYPES",
        "ATTRIBUTES",
        "IS_ORIGINAL_TITLE",
        "IS_ORIGINAL_TITLE_FLAG",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
@dlt.table(
    name="silver.silver_title_language_codes",
    comment="Cleaned IMDB language codes lookup table - preserves all records",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
def silver_title_language_codes():
    df = dlt.read_stream("bronze_title_language_codes_raw")
    
    # # Replace \N with NULLs
    # df = df.select(
    #     *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    # )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("Language_Name", "LANGUAGE_NAME")
        .withColumnRenamed("Language_CODE", "LANGUAGE_CODE")
    )
    
    # # Handle NULL values
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", 
    #                when(col("LANGUAGE_NAME").isNull(), "Unknown")
    #                .otherwise(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", 
    #                when(col("LANGUAGE_CODE").isNull(), "unknown")
    #                .otherwise(col("LANGUAGE_CODE")))
    # )
    
    # # Trim whitespace
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", trim(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", trim(col("LANGUAGE_CODE")))
    # )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "LANGUAGE_NAME",
        "LANGUAGE_CODE",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
@dlt.table(
    name="silver.silver_title_ratings",
    comment="Silver layer - Clean IMDb ratings with Decimal(3,1) precision and rating category."
)
@dlt.expect_or_drop("tconst_not_null", "tconst IS NOT NULL")
def title_ratings_silver():
    df = (
        dlt.read_stream("bronze_title_ratings_raw")
       
        # Remove rescued data if exists
        .drop("_rescued_data")
       
        # String type for identifiers
        .withColumn("tconst", col("tconst").cast(StringType()))
       
        # Using decimal(3,1) for averageRating
        .withColumn("averageRating",
            round(col("averageRating").cast("double"), 1).cast("double")
        )
       
        # Derived column: Rating Category based on averageRating
        .withColumn("rating_category",
            when(col("averageRating") <= 2.0, "Poor")
            .when(col("averageRating") <= 4.0, "Below Average")
            .when(col("averageRating") <= 6.0, "Average")
            .when(col("averageRating") <= 8.0, "Good")
            .when(col("averageRating") <= 10.0, "Excellent")
            .otherwise("Unknown")
        )
       
        # Casting for integer votes
        .withColumn("numVotes", col("numVotes").cast(IntegerType()))
       
        # Audit column
        .withColumn("silver_load_dt", current_timestamp())
    )
        # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])
   
    return df

In [0]:
@dlt.table(
    name="silver_title_basics",
    comment="Cleaned title basics with genres as array - IS_ADULT: 1=Adult, 0=Not Adult, -1=Unknown",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all({
    "valid_year_range": "START_YEAR <= END_YEAR",
    "reasonable_runtime": "RUNTIME_MINUTES < 50000"
})
@dlt.expect_or_drop("tconst_not_null", "TCONST IS NOT NULL")
@dlt.expect_or_drop("valid_tconst", "TCONST RLIKE '^tt[0-9]{7,8}$'")
def silver_title_basics():
    """Silver transformation for title.basics with UPPERCASE column names
    
    IS_ADULT encoding:
    - 1: Adult content
    - 0: Not adult content  
    - -1: Unknown/Invalid data
    """
    
    df = (
        dlt.read_stream("bronze_title_basics_raw")  
        .drop("_rescued_data")
        
        # --- String columns: Cast and trim ---
        .withColumn("TCONST", col("tconst").cast("string"))
        .withColumn("TITLE_TYPE", col("titleType").cast("string"))
        .withColumn("PRIMARY_TITLE", trim(col("primaryTitle")).cast("string"))
        .withColumn("ORIGINAL_TITLE", trim(col("originalTitle")).cast("string"))
        
        # --- isAdult: Convert to Integer (1=adult, 0=not adult, -1=unknown) ---
        .withColumn("IS_ADULT",
            when(col("isAdult") == "1", 1)
            .when(col("isAdult") == "0", 0)
            .otherwise(-1)  # Invalid values like '2019', '\\N', or anything else
            .cast("int")
        )
        
        # --- startYear: Replace NULL with 0, cast to Integer ---
        .withColumn("START_YEAR",
            when(col("startYear").isNull(), 0)
            .otherwise(col("startYear").cast("int"))
        )
        
        # --- endYear: Replace NULL with 9999, cast to Integer ---
        .withColumn("END_YEAR",
            when(col("endYear").isNull(), 9999)
            .otherwise(col("endYear").cast("int"))
        )
        
        # --- runtimeMinutes: Replace NULL with 0, cast to Integer ---
        .withColumn("RUNTIME_MINUTES",
            when(col("runtimeMinutes").isNull(), 0)
            .otherwise(col("runtimeMinutes").cast("int"))
        )
        
        # --- genres: Replace NULL with 'unknown' ---
        .withColumn("GENRES",
            when(col("genres").isNull(), "unknown")
            .otherwise(col("genres"))
        )
        
        # --- Create genre array ---
        .withColumn("GENRE_ARRAY", split(col("GENRES"), ","))
        
        # --- Add silver metadata ---
        .withColumn("SILVER_PROCESSING_TIMESTAMP", current_timestamp())
    )
    
    return df.select(
        "TCONST",
        "TITLE_TYPE",
        "PRIMARY_TITLE",
        "ORIGINAL_TITLE",
        "IS_ADULT",
        "START_YEAR",
        "END_YEAR",
        "RUNTIME_MINUTES",
        "GENRE_ARRAY",
        "INGESTION_TIMESTAMP",
        "SILVER_PROCESSING_TIMESTAMP",
        "SOURCE_FILE",
        "INGESTION_DATE"
    )

In [0]:
import dlt
from pyspark.sql.functions import *
 
@dlt.table(
    name="silver_title_crew",
    comment="Cleaned and exploded title crew data - one row per crew member (excluding Unknown)",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "TCONST IS NOT NULL AND TCONST RLIKE '^tt[0-9]{7,8}$'",
    "valid_crew_member": "NCONST != 'Unknown'"
})
@dlt.expect_all_or_fail({
    "tconst_not_null": "TCONST IS NOT NULL",
    "tconst_not_empty": "LENGTH(TCONST) >= 9"
})
def silver_title_crew():
    """Silver transformation for title.crew with exploded crew members - UPPERCASE columns"""
    
    df = dlt.read_stream("bronze_title_crew_raw")
    
    # STEP 1: Replace \N with NULL
    df = df.select(
        col("tconst").alias("TCONST"),
        when(col("directors") == "\\N", None).otherwise(col("directors")).alias("DIRECTORS"),
        when(col("writers") == "\\N", None).otherwise(col("writers")).alias("WRITERS"),
        col("ingestion_timestamp").alias("INGESTION_TIMESTAMP"),
        col("source_file").alias("SOURCE_FILE"),
        col("ingestion_date").alias("INGESTION_DATE")
    )
    
    # STEP 2: TRIM Whitespace
    df = (
        df
        .withColumn("DIRECTORS",
                   when(col("DIRECTORS").isNotNull(), trim(col("DIRECTORS")))
                   .otherwise(None))
        .withColumn("WRITERS",
                   when(col("WRITERS").isNotNull(), trim(col("WRITERS")))
                   .otherwise(None))
    )
    
    # STEP 3: FILTER OUT rows where BOTH directors AND writers are NULL
    df = df.filter(col("DIRECTORS").isNotNull() | col("WRITERS").isNotNull())
    
    # STEP 4: Create arrays from comma-separated strings (NO "Unknown" fallback)
    df = (
        df
        .withColumn("DIRECTOR_ARRAY",
                   when(col("DIRECTORS").isNotNull(), split(col("DIRECTORS"), ","))
                   .otherwise(array()))
        .withColumn("WRITER_ARRAY",
                   when(col("WRITERS").isNotNull(), split(col("WRITERS"), ","))
                   .otherwise(array()))
    )
    
    # STEP 5: Explode directors into separate rows (only if array is not empty)
    df_directors = (
        df
        .filter(size(col("DIRECTOR_ARRAY")) > 0)
        .select(
            "TCONST",
            explode("DIRECTOR_ARRAY").alias("NCONST"),
            "INGESTION_TIMESTAMP",
            "SOURCE_FILE",
            "INGESTION_DATE"
        )
        .withColumn("NCONST", trim(col("NCONST")))
        .withColumn("CREW_ROLE", lit("director"))
    )
    
    # STEP 6: Explode writers into separate rows (only if array is not empty)
    df_writers = (
        df
        .filter(size(col("WRITER_ARRAY")) > 0)
        .select(
            "TCONST",
            explode("WRITER_ARRAY").alias("NCONST"),
            "INGESTION_TIMESTAMP",
            "SOURCE_FILE",
            "INGESTION_DATE"
        )
        .withColumn("NCONST", trim(col("NCONST")))
        .withColumn("CREW_ROLE", lit("writer"))
    )
    
    # STEP 7: Union directors and writers
    df_exploded = df_directors.union(df_writers)
    
    # STEP 8: Add data quality flags (commented out)
    # df_exploded = (
    #     df_exploded
    #     .withColumn("IS_UNKNOWN_CREW", col("NCONST") == "Unknown")
    #     .withColumn("HAS_VALID_CREW_ID",
    #                col("NCONST").rlike("^nm[0-9]{7,8}$"))
    #     .withColumn("DATA_QUALITY_TIER",
    #                when(col("IS_UNKNOWN_CREW"), "Unknown_Crew")
    #                .when(~col("HAS_VALID_CREW_ID"), "Invalid_Crew_ID")
    #                .otherwise("Complete"))
    # )
    
    # STEP 9: ADD SILVER METADATA
    df_exploded = df_exploded.withColumn("SILVER_PROCESSING_TIMESTAMP", current_timestamp())
    
    return df_exploded

In [0]:
@dlt.table(
    name="silver.silver_title_principals",
    comment="Silver layer - Cleaned principals data. Nulls handled, data types standardized, whitespace trimmed.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "tconst_not_null": "tconst IS NOT NULL",
    "nconst_not_null": "nconst IS NOT NULL"
})
def silver_title_principals():
    df = dlt.read_stream("bronze_title_principals_raw")
    
    # Remove rescued data if exists
    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")
    
    # Handle NULL values for string columns
    df = (
        df
        .withColumn("tconst", 
                   when(col("tconst").isNull(), "unknown")
                   .otherwise(col("tconst")))
        .withColumn("nconst", 
                   when(col("nconst").isNull(), "unknown")
                   .otherwise(col("nconst")))
        .withColumn("category", 
                   when(col("category").isNull(), "unknown")
                   .otherwise(col("category")))
        .withColumn("job", 
                   when(col("job").isNull(), "unknown")
                   .otherwise(col("job")))
        .withColumn("characters", 
                   when(col("characters").isNull(), "unknown")
                   .otherwise(col("characters")))
    )
    
    # Handle NULL values for numeric columns
    df = df.withColumn("ordering", 
                      when(col("ordering").isNull(), -1)
                      .otherwise(col("ordering")))
    
    # Cast to appropriate types
    df = (
        df
        .withColumn("tconst", col("tconst").cast(StringType()))
        .withColumn("ordering", col("ordering").cast(IntegerType()))
        .withColumn("nconst", col("nconst").cast(StringType()))
        .withColumn("category", col("category").cast(StringType()))
        .withColumn("job", col("job").cast(StringType()))
        .withColumn("characters", col("characters").cast(StringType()))
    )
    
    # Trim whitespace
    df = (
        df
        .withColumn("job", trim(col("job")))
        .withColumn("characters", trim(col("characters")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_load_dt", 
        current_timestamp()
    )
    
        # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])
    
    return df


In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_title_episode",
    comment="Cleaned IMDB title episode data with NULL handling and type conversions",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "tconst IS NOT NULL AND tconst RLIKE '^tt[0-9]{7,}$'",
    "valid_parent_tconst": "parentTconst IS NOT NULL AND parentTconst RLIKE '^tt[0-9]{7,}$'"
})
def silver_title_episode():
    """
    Cleans the bronze title_episode data.
    
    Transformations:
    - Replace \\N with NULLs
    - Replace NULL seasonNumber with -1
    - Replace NULL episodeNumber with -1
    - Cast to appropriate types
    - Validate tconst and parentTconst formats
    """
    
    df = dlt.read_stream("bronze_title_episode_raw")
    
    # Drop rescued data if exists
    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("seasonNumber", 
                   when(col("seasonNumber").isNull(), "-1")
                   .otherwise(col("seasonNumber")))
        .withColumn("episodeNumber", 
                   when(col("episodeNumber").isNull(), "-1")
                   .otherwise(col("episodeNumber")))
    )
    
    # Cast to appropriate types
    df = (
        df
        .withColumn("tconst", col("tconst").cast("string"))
        .withColumn("parentTconst", col("parentTconst").cast("string"))
        .withColumn("seasonNumber", col("seasonNumber").cast("int"))
        .withColumn("episodeNumber", col("episodeNumber").cast("int"))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )

        # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])
    
    # Select final columns
    return df.select(
        "tconst",
        "parentTconst",
        "seasonNumber",
        "episodeNumber",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )


In [0]:
@dlt.table(
    name="silver.silver_title_region",
    comment="IMDB title.region lookup data - pass-through from bronze",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "tconst IS NOT NULL"
})
def silver_title_region():
    """
    Pass-through table from bronze to silver.
    
    Transformations:
    - No data transformations applied
    - Add silver processing timestamp for lineage tracking
    """
    
    df = dlt.read_stream("bronze_title_region_raw")
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "tconst",
        "region",
        "country_code_country_name",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )
